In [39]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision import transforms
import timm

In [40]:
# =========================================
# IMPORT DATASET
# =========================================
sys.path.append(os.path.abspath(".."))
from scripts.Loading_Dataset import LiverDataset

In [41]:
# =========================================
# DEVICE
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [42]:
# =========================================
# TRANSFORMS (IMPORTANT FIX)
# PIL → Tensor → ViT compatible
# =========================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [43]:
# =========================================
# DATASET
# =========================================
root_dir = "../Dataset"

dataset = LiverDataset(
    root_dir=root_dir,
    transform=transform
)


In [44]:
# =========================================
# DATALOADER
# =========================================
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [45]:
# =========================================
# VISION TRANSFORMER (ViT)
# =========================================
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=0   # returns CLS embedding
)

model = model.to(device)
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [ ]:

# =========================================
# FEATURE EXTRACTION
# =========================================
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Extracting ViT features"):

        # move to GPU
        images = images.to(device)

        # forward pass → [B, 768]
        features = model(images)

        # store results
        all_features.append(features.cpu())
        all_labels.append(labels)

Extracting ViT features:   3%|▎         | 3/112 [01:38<50:33, 27.83s/it]  